# Sequential Agent Pipelines — Chains of Specialized Agents

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/19_sequential_agent_pipelines.ipynb)

## What This Notebook Teaches You

A single LLM call tries to do everything at once: research, verify, summarize, format.
That's like asking one factory worker to mine ore, smelt steel, machine parts, and assemble a car.
**Pipelines** split complex work into specialized stages, each performed by a focused agent.

**Research foundation**:
- Wu et al. 2023, *"AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation"* — demonstrated that multi-agent conversation patterns (including sequential handoff) outperform single-agent approaches
- Chase 2022, *"LangChain: Building applications with LLMs through composability"* — introduced the chain abstraction for sequencing LLM calls
- Weng 2023, *"LLM Powered Autonomous Agents"* — surveyed pipeline and planning approaches in agent systems

By the end of this notebook, you will:

1. **Understand why pipelines outperform monolithic agents** — specialization, error isolation, debuggability
2. **Build a Message dataclass** for standardized inter-agent communication
3. **Build AgentNode and Pipeline classes** from scratch
4. **Run two complete pipelines** — Research and Content — and inspect intermediate outputs
5. **Implement advanced features** — partial execution, retry, conditional branching
6. **Know when pipelines help and when they add unnecessary overhead**

---

> **Prerequisites**: Notebooks 01–06 (agent fundamentals, tool use, ReAct, prompt chaining).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~55–70 minutes.

## Part 0: Environment Setup

Same Qwen3-8B model. Each pipeline stage will use a **different system prompt** to specialize the model's behavior — the same LLM plays different roles depending on which stage it's in.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Pipelines?

### 1.1 — The Unix Pipes Analogy

Unix became powerful not because of any single tool, but because small, specialized tools could be **chained together**:

```bash
cat access.log | grep "ERROR" | sort | uniq -c | sort -rn | head -10
```

Each tool does **one thing well** and passes its output to the next. LLM agent pipelines work the same way:

```
[Researcher] → [Fact-Checker] → [Summarizer] → Final Output
```

### 1.2 — Why Specialized Agents Outperform Generalists

| Aspect | Single Agent (Monolithic) | Pipeline (Specialized) |
|---|---|---|
| **Prompt length** | Huge — must include all instructions | Short — each stage has focused instructions |
| **Error isolation** | One mistake contaminates everything | Errors caught at stage boundaries |
| **Debugging** | "Which part went wrong?" Hard to tell | Inspect intermediate outputs at each stage |
| **Reliability** | All-or-nothing | Partial results still useful |
| **Token efficiency** | Wasted context on irrelevant instructions | Each agent uses context only for its task |

### 1.3 — The Assembly Line Analogy

In manufacturing, Henry Ford's assembly line showed that **specialized workers at stations** outperform generalists:
- Each station has specific tools and expertise
- Quality checks happen between stations
- A defective part gets caught before reaching the next station
- The whole line is faster than one person building a complete car

Agent pipelines apply this same principle to knowledge work.

## 2. The Message Dataclass — Standardized Communication

Before building agents, we need a **standard envelope** for data passing between stages. Without it, each agent would need custom parsing logic for every predecessor's output format.

In [ ]:
from datetime import datetime

@dataclass
class Message:
    """Standard format for inter-agent communication.

    Why a dataclass instead of raw strings?
    1. Metadata travels WITH the content (no lost context)
    2. Pipeline can inspect source_agent and timestamp for logging
    3. Type safety: code that expects a Message won't accept a random string
    """
    content: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    source_agent: str = "user"
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

    def summary(self, max_len: int = 120) -> str:
        """Short preview for logging."""
        preview = self.content[:max_len].replace("\n", " ")
        if len(self.content) > max_len:
            preview += "..."
        return f"[{self.source_agent} @ {self.timestamp[:19]}] {preview}"

    def with_metadata(self, **kwargs) -> 'Message':
        """Return a copy with additional metadata fields."""
        new_meta = {**self.metadata, **kwargs}
        return Message(
            content=self.content,
            metadata=new_meta,
            source_agent=self.source_agent,
            timestamp=self.timestamp,
        )

# Test it
msg = Message(content="What are the main causes of climate change?", source_agent="user")
print(msg.summary())
print(f"Metadata: {msg.metadata}")

tagged = msg.with_metadata(priority="high", domain="science")
print(f"Tagged: {tagged.metadata}")

## 3. The AgentNode Class — A Specialized Pipeline Stage

Each AgentNode wraps a system prompt and a role name. When it processes a message, it calls the LLM with its specialized prompt, then wraps the result in a new Message.

```
                 ┌────────────────────────────┐
  Input Message  │        AgentNode           │  Output Message
 ───────────────►│  system_prompt = "You are  │──────────────────►
                 │  a fact-checker..."        │
                 │  role = "fact_checker"     │
                 └────────────────────────────┘
```

In [ ]:
@dataclass
class AgentNode:
    """A single specialized agent in a pipeline.

    Design decisions:
    - system_prompt defines the agent's expertise and behavior
    - role is a short identifier used in logs and Message.source_agent
    - max_tokens controls output length (different stages need different lengths)
    - temperature controls creativity (fact-checkers should be low, writers higher)
    """
    role: str
    system_prompt: str
    max_tokens: int = 512
    temperature: float = 0.7

    def process(self, input_message: Message) -> Message:
        """Process an input message and return an output message."""
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": input_message.content},
        ]

        start = time.time()
        response = generate(
            messages,
            max_new_tokens=self.max_tokens,
            temperature=self.temperature,
        )
        elapsed = time.time() - start

        return Message(
            content=response,
            metadata={
                **input_message.metadata,
                "processing_time": round(elapsed, 2),
                "stage": self.role,
                "input_length": len(input_message.content),
                "output_length": len(response),
            },
            source_agent=self.role,
        )

    def __repr__(self):
        return f"AgentNode(role='{self.role}')"

# Test with a simple agent
test_agent = AgentNode(
    role="greeter",
    system_prompt="You are a friendly greeter. Respond with a brief, warm greeting.",
    max_tokens=100,
)

test_msg = Message(content="Hello, I'm new here!", source_agent="user")
result = test_agent.process(test_msg)
print(f"Input:  {test_msg.summary()}")
print(f"Output: {result.summary()}")
print(f"Processing time: {result.metadata['processing_time']}s")

## 4. The Pipeline Class — Chaining Agents Together

The Pipeline class is the orchestrator. It takes a list of AgentNodes and runs them in sequence, passing each stage's output as the next stage's input.

```
  Input ──► [Stage 1] ──► [Stage 2] ──► [Stage 3] ──► Output
                │              │              │
                ▼              ▼              ▼
           intermediate   intermediate   intermediate
            result 1       result 2       result 3
```

Key design goals:
- **Logging**: record every intermediate message for debugging
- **Error propagation**: if a stage fails, capture the error and decide whether to continue
- **Partial execution**: ability to run only the first N stages

In [ ]:
@dataclass
class PipelineResult:
    """Complete result from a pipeline run."""
    final_output: Optional[Message] = None
    intermediate_results: List[Message] = field(default_factory=list)
    errors: List[Dict[str, Any]] = field(default_factory=list)
    total_time: float = 0.0
    stages_completed: int = 0

    def show_flow(self):
        """Print the flow of messages through the pipeline."""
        print("=" * 70)
        print("PIPELINE EXECUTION FLOW")
        print("=" * 70)
        for i, msg in enumerate(self.intermediate_results):
            stage_label = f"Stage {i}" if i > 0 else "Input"
            print(f"\n{'─' * 70}")
            print(f"  {stage_label}: [{msg.source_agent}]")
            if 'processing_time' in msg.metadata:
                print(f"  Time: {msg.metadata['processing_time']}s | "
                      f"Output length: {msg.metadata.get('output_length', 'N/A')} chars")
            print(f"{'─' * 70}")
            print(msg.content[:500])
            if len(msg.content) > 500:
                print(f"  ... ({len(msg.content) - 500} more chars)")
        if self.errors:
            print(f"\n⚠ Errors: {len(self.errors)}")
            for e in self.errors:
                print(f"  - Stage '{e['stage']}': {e['error']}")
        print(f"\n{'=' * 70}")
        print(f"Total time: {self.total_time:.2f}s | Stages completed: {self.stages_completed}")
        print("=" * 70)


class Pipeline:
    """Sequential pipeline that chains AgentNodes.

    Why a class and not just a for-loop?
    - Centralized logging and error handling
    - Partial execution support
    - Retry logic per stage
    - Extensible (conditional branching, callbacks)
    """

    def __init__(self, name: str, stages: List[AgentNode]):
        self.name = name
        self.stages = stages

    def run(
        self,
        input_message: Message,
        max_stages: Optional[int] = None,
        max_retries: int = 1,
    ) -> PipelineResult:
        """Execute the pipeline.

        Args:
            input_message: Starting message
            max_stages: Run only first N stages (None = all)
            max_retries: How many times to retry a failed stage
        """
        result = PipelineResult()
        result.intermediate_results.append(input_message)
        current = input_message
        stages_to_run = self.stages[:max_stages] if max_stages else self.stages
        pipeline_start = time.time()

        print(f"🔗 Pipeline '{self.name}' starting ({len(stages_to_run)} stages)")
        for i, stage in enumerate(stages_to_run):
            print(f"  ▶ Stage {i + 1}/{len(stages_to_run)}: {stage.role}...", end=" ")
            attempts = 0
            success = False

            while attempts < max_retries and not success:
                try:
                    current = stage.process(current)
                    result.intermediate_results.append(current)
                    result.stages_completed += 1
                    success = True
                    print(f"✓ ({current.metadata.get('processing_time', '?')}s)")
                except Exception as e:
                    attempts += 1
                    if attempts >= max_retries:
                        error_info = {"stage": stage.role, "error": str(e), "attempt": attempts}
                        result.errors.append(error_info)
                        print(f"✗ (failed after {attempts} attempt(s))")
                        break
                    print(f"↻ retry {attempts}...", end=" ")

        result.total_time = time.time() - pipeline_start
        result.final_output = current
        print(f"🏁 Pipeline complete in {result.total_time:.2f}s")
        return result

    def __repr__(self):
        stage_names = " → ".join(s.role for s in self.stages)
        return f"Pipeline('{self.name}': {stage_names})"

print("✓ Pipeline infrastructure ready")

## 5. Example Pipeline 1: Research Pipeline

Our first real pipeline performs a classic knowledge task:

```
[Researcher] ──► [Fact-Checker] ──► [Summarizer]
    │                  │                  │
    │ Find info        │ Verify claims    │ Condense to
    │ on a topic       │ and flag errors  │ key points
```

Each agent sees ONLY the output of the previous stage — not the original question. This forces clean interfaces.

In [ ]:
# Define the three specialized agents
researcher = AgentNode(
    role="researcher",
    system_prompt="""You are a thorough research analyst. Your job:
1. Gather comprehensive information on the given topic
2. Cover multiple angles and perspectives
3. Include specific facts, figures, and examples where possible
4. Organize information clearly with headers and bullet points
5. Cite the general sources of your knowledge (e.g., "According to WHO data...")

Output ONLY your research findings. Do not summarize or conclude — later stages will do that.""",
    max_tokens=700,
    temperature=0.7,
)

fact_checker = AgentNode(
    role="fact_checker",
    system_prompt="""You are a meticulous fact-checker. Your job:
1. Read the research text provided
2. Identify each factual claim made
3. For each claim, assess if it's: VERIFIED (well-known fact), PLAUSIBLE (reasonable but unverified), or QUESTIONABLE (potentially wrong)
4. Flag any contradictions or logical errors
5. Output the CORRECTED version of the research, keeping verified facts and fixing or flagging questionable ones

Format:
- Start with "FACT-CHECK SUMMARY:" listing claims and their status
- Then provide the "CORRECTED TEXT:" with fixes applied""",
    max_tokens=700,
    temperature=0.3,  # Low temperature for factual accuracy
)

summarizer = AgentNode(
    role="summarizer",
    system_prompt="""You are an expert summarizer. Your job:
1. Take the fact-checked research and distill it to key points
2. Preserve important facts and figures
3. Remove redundancy and verbose explanations
4. Produce a concise, well-structured summary (3-5 key points)
5. End with a one-sentence takeaway

Keep the summary under 200 words. Prioritize accuracy over completeness.""",
    max_tokens=400,
    temperature=0.5,
)

# Assemble the pipeline
research_pipeline = Pipeline(
    name="Research Pipeline",
    stages=[researcher, fact_checker, summarizer],
)

print(research_pipeline)

### 5.1 — Running the Research Pipeline

Let's test it with a real research question and inspect every intermediate output.

In [ ]:
# Run the research pipeline
research_input = Message(
    content="What are the main health effects of microplastics in drinking water? "
            "Cover what is currently known from scientific research.",
    source_agent="user",
)

research_result = research_pipeline.run(research_input)
research_result.show_flow()

## 6. Example Pipeline 2: Content Pipeline

The content pipeline has **four stages** — more specialization, more refinement:

```
[Drafter] ──► [Editor] ──► [Formatter] ──► [Quality-Checker]
    │              │             │                │
    │ Raw draft    │ Improve     │ Structure &    │ Final quality
    │ of content   │ clarity &   │ format for     │ score and
    │              │ flow        │ readability    │ pass/fail
```

Watch how quality improves at each stage.

In [ ]:
# Define four specialized agents
drafter = AgentNode(
    role="drafter",
    system_prompt="""You are a content drafter. Write a first draft on the given topic.
Focus on getting ideas down — don't worry about perfect prose.
Include all key points even if rough. Aim for 200-300 words.
Write in a conversational, accessible style suitable for a general audience.""",
    max_tokens=500,
    temperature=0.8,  # Higher creativity for drafting
)

editor = AgentNode(
    role="editor",
    system_prompt="""You are a professional editor. Your job:
1. Read the draft and improve clarity, flow, and readability
2. Fix grammatical errors and awkward phrasing
3. Strengthen weak arguments or vague statements
4. Ensure logical flow between paragraphs
5. Keep the author's voice — don't rewrite entirely

Output the EDITED version of the text with your improvements applied.
Add a brief "EDITOR'S NOTE:" at the end listing the 3 most important changes you made.""",
    max_tokens=600,
    temperature=0.5,
)

formatter = AgentNode(
    role="formatter",
    system_prompt="""You are a document formatter and structure specialist. Your job:
1. Take the edited text and add clear structure
2. Add a compelling title if missing
3. Break into sections with descriptive headers
4. Use bullet points or numbered lists where appropriate
5. Add a brief introduction and conclusion if missing
6. Ensure the text is scannable — a reader should grasp key points in 30 seconds

Output the FORMATTED version. Do not change the content — only improve structure and presentation.""",
    max_tokens=600,
    temperature=0.3,
)

quality_checker = AgentNode(
    role="quality_checker",
    system_prompt="""You are a quality assurance reviewer. Evaluate the content on these dimensions:
1. ACCURACY: Are claims reasonable and well-supported?
2. CLARITY: Is the text clear and understandable?
3. STRUCTURE: Is the text well-organized?
4. COMPLETENESS: Are key aspects of the topic covered?
5. ENGAGEMENT: Is the text interesting to read?

Score each dimension 1-5 and give an overall PASS/FAIL verdict.

Format:
QUALITY REPORT:
- Accuracy: X/5 — [brief note]
- Clarity: X/5 — [brief note]
- Structure: X/5 — [brief note]
- Completeness: X/5 — [brief note]
- Engagement: X/5 — [brief note]
Overall: X/25 — [PASS if >= 18, FAIL otherwise]
[Brief final comment]""",
    max_tokens=400,
    temperature=0.3,
)

content_pipeline = Pipeline(
    name="Content Pipeline",
    stages=[drafter, editor, formatter, quality_checker],
)

print(content_pipeline)

### 6.1 — Running the Content Pipeline

In [ ]:
content_input = Message(
    content="Write an article explaining why sleep is important for learning and memory. "
            "Target audience: college students who often sacrifice sleep for studying.",
    source_agent="user",
)

content_result = content_pipeline.run(content_input)
content_result.show_flow()

## 7. Pipeline Features — Going Beyond Simple Chaining

### 7.1 — Partial Execution

Sometimes you want to run only part of the pipeline — perhaps to inspect intermediate output or because later stages are expensive.

In [ ]:
# Partial execution: run only the first 2 stages of the content pipeline
partial_input = Message(
    content="Explain the double-slit experiment in quantum physics for a curious teenager.",
    source_agent="user",
)

partial_result = content_pipeline.run(partial_input, max_stages=2)
print(f"Stages completed: {partial_result.stages_completed} of {len(content_pipeline.stages)}")
print(f"\nDraft + Edit output (no formatting or QA):")
print(partial_result.final_output.content[:600])

### 7.2 — Stage Retry on Failure

If a stage fails (e.g., the model returns garbage or an API error occurs), the pipeline retries before giving up. We can also build a **validation wrapper** that checks output quality.

In [ ]:
class ValidatedAgentNode(AgentNode):
    """AgentNode with output validation.

    If the validator returns False, the output is treated as a failure
    and the stage retries (if retries are configured in the pipeline).
    """

    def __init__(self, validator: Callable[[str], bool] = None, **kwargs):
        super().__init__(**kwargs)
        self.validator = validator

    def process(self, input_message: Message) -> Message:
        result = super().process(input_message)
        if self.validator and not self.validator(result.content):
            raise ValueError(
                f"Validation failed for stage '{self.role}': "
                f"output did not pass quality check"
            )
        return result

# Example: a summarizer that MUST produce output under 300 words
def length_validator(text: str) -> bool:
    word_count = len(text.split())
    return word_count <= 300

validated_summarizer = ValidatedAgentNode(
    validator=length_validator,
    role="validated_summarizer",
    system_prompt="""Summarize the given text in under 200 words.
Be concise and focus on key points only. Do not exceed 200 words.""",
    max_tokens=350,
    temperature=0.4,
)

# Test the validated node
test_input = Message(content="The Earth revolves around the Sun. This was discovered centuries ago.", source_agent="user")
validated_output = validated_summarizer.process(test_input)
word_count = len(validated_output.content.split())
print(f"Output word count: {word_count} (limit: 300)")
print(f"Validation would pass: {length_validator(validated_output.content)}")

### 7.3 — Intermediate Result Inspection

One of the biggest advantages of pipelines over monolithic agents: you can **inspect and analyze** every intermediate result. This is invaluable for debugging and understanding agent behavior.

In [ ]:
def analyze_pipeline_result(result: PipelineResult):
    """Analyze how content transforms through pipeline stages."""
    print("📊 PIPELINE ANALYSIS")
    print("=" * 60)

    for i, msg in enumerate(result.intermediate_results):
        label = "Input" if i == 0 else f"Stage {i}: {msg.source_agent}"
        words = len(msg.content.split())
        chars = len(msg.content)
        sentences = msg.content.count('.') + msg.content.count('!') + msg.content.count('?')
        proc_time = msg.metadata.get('processing_time', 0)

        print(f"\n{label}:")
        print(f"  Words: {words:,} | Chars: {chars:,} | Sentences: ~{sentences}")
        if proc_time:
            print(f"  Processing time: {proc_time}s")

    # Show content length evolution
    lengths = [len(m.content.split()) for m in result.intermediate_results]
    print(f"\n📈 Word count evolution: {' → '.join(str(l) for l in lengths)}")

    # Total processing
    total_tokens_estimate = sum(
        m.metadata.get('input_length', 0) + m.metadata.get('output_length', 0)
        for m in result.intermediate_results
    )
    print(f"⏱ Total pipeline time: {result.total_time:.2f}s")
    print(f"📝 Est. total chars processed: {total_tokens_estimate:,}")

# Analyze the research pipeline result
analyze_pipeline_result(research_result)

## 8. Conditional Pipelines — Dynamic Routing

Not all content should flow through the same stages. A **ConditionalPipeline** routes messages to different next stages based on the output content.

```
                        ┌──► [Technical Editor] ──► [Formatter]
                        │
[Classifier] ──► Route ─┤
                        │
                        └──► [Simple Editor] ──► [Formatter]
```

In [ ]:
class ConditionalPipeline:
    """Pipeline with conditional branching based on stage output.

    Each stage can specify a 'router' function that determines which
    stage to run next based on the current message content.
    """

    def __init__(self, name: str):
        self.name = name
        self.stages: Dict[str, AgentNode] = {}
        self.routes: Dict[str, Callable[[Message], str]] = {}
        self.start_stage: Optional[str] = None

    def add_stage(self, agent: AgentNode, next_stage: str = None,
                  router: Callable[[Message], str] = None):
        """Add a stage with either a fixed next stage or a dynamic router."""
        self.stages[agent.role] = agent
        if not self.start_stage:
            self.start_stage = agent.role
        if router:
            self.routes[agent.role] = router
        elif next_stage:
            self.routes[agent.role] = lambda msg, ns=next_stage: ns

    def run(self, input_message: Message, max_steps: int = 10) -> PipelineResult:
        """Execute the conditional pipeline."""
        result = PipelineResult()
        result.intermediate_results.append(input_message)
        current = input_message
        current_stage_name = self.start_stage
        pipeline_start = time.time()

        print(f"🔀 Conditional Pipeline '{self.name}' starting")
        step = 0
        while current_stage_name and step < max_steps:
            stage = self.stages.get(current_stage_name)
            if not stage:
                print(f"  ⚠ Stage '{current_stage_name}' not found — stopping")
                break

            print(f"  ▶ Step {step + 1}: {stage.role}...", end=" ")
            try:
                current = stage.process(current)
                result.intermediate_results.append(current)
                result.stages_completed += 1
                print(f"✓")
            except Exception as e:
                result.errors.append({"stage": stage.role, "error": str(e)})
                print(f"✗ ({e})")
                break

            # Determine next stage
            router = self.routes.get(current_stage_name)
            current_stage_name = router(current) if router else None
            step += 1

        result.total_time = time.time() - pipeline_start
        result.final_output = current
        print(f"🏁 Done in {result.total_time:.2f}s ({result.stages_completed} stages)")
        return result

print("✓ ConditionalPipeline ready")

### 8.1 — Building a Conditional Content Pipeline

Let's build a pipeline that classifies content complexity first, then routes to either a technical or simple editing path.

In [ ]:
# Classifier determines the content type
classifier = AgentNode(
    role="classifier",
    system_prompt="""Classify the given text into exactly one category.
Respond with ONLY one word — either TECHNICAL or SIMPLE.
- TECHNICAL: contains jargon, scientific terms, complex concepts, code, or data
- SIMPLE: conversational, everyday language, general audience

Your response must be exactly one word: TECHNICAL or SIMPLE""",
    max_tokens=10,
    temperature=0.1,
)

# Two different editor paths
technical_editor = AgentNode(
    role="technical_editor",
    system_prompt="""You are a technical editor. The text contains specialized content.
1. Ensure technical accuracy of terminology
2. Add clarifying explanations for complex concepts
3. Keep precise language — don't oversimplify
4. Add brief definitions for jargon on first use
Output the edited version.""",
    max_tokens=600,
    temperature=0.4,
)

simple_editor = AgentNode(
    role="simple_editor",
    system_prompt="""You are an editor for general audiences. The text is conversational.
1. Improve readability and flow
2. Make it engaging and relatable
3. Use active voice and clear sentences
4. Add examples or analogies where helpful
Output the edited version.""",
    max_tokens=600,
    temperature=0.6,
)

final_formatter = AgentNode(
    role="final_formatter",
    system_prompt="""Format the text with a clear title, sections, and conclusion.
Make it scannable with headers and bullet points where appropriate.
Output the final formatted version.""",
    max_tokens=600,
    temperature=0.3,
)

# Build the conditional pipeline
def route_by_complexity(msg: Message) -> str:
    content_upper = msg.content.strip().upper()
    if "TECHNICAL" in content_upper:
        return "technical_editor"
    return "simple_editor"

cond_pipeline = ConditionalPipeline(name="Smart Content Pipeline")
cond_pipeline.add_stage(classifier, router=route_by_complexity)
cond_pipeline.add_stage(technical_editor, next_stage="final_formatter")
cond_pipeline.add_stage(simple_editor, next_stage="final_formatter")
cond_pipeline.add_stage(final_formatter)  # Terminal stage — no next

# Test with technical content
tech_input = Message(
    content="Explain how CRISPR-Cas9 gene editing works, including the guide RNA mechanism, "
            "double-strand break repair pathways (NHEJ vs HDR), and recent developments in "
            "base editing and prime editing.",
    source_agent="user",
)

print("--- Technical content test ---")
tech_result = cond_pipeline.run(tech_input)

print(f"\nRoute taken: {' → '.join(m.source_agent for m in tech_result.intermediate_results)}")
print(f"\nFinal output preview:\n{tech_result.final_output.content[:400]}")

### 8.2 — Testing with Simple Content

In [ ]:
# Test with simple content — should route differently
simple_input = Message(
    content="Write a paragraph about why dogs make great pets. "
            "Keep it friendly and fun for a family audience.",
    source_agent="user",
)

print("--- Simple content test ---")
simple_result = cond_pipeline.run(simple_input)

print(f"\nRoute taken: {' → '.join(m.source_agent for m in simple_result.intermediate_results)}")
print(f"\nFinal output preview:\n{simple_result.final_output.content[:400]}")

## 9. Comparison: Single Agent vs. Pipeline

The critical question: **does a pipeline actually improve quality?** Let's compare three approaches on the same task:

1. **Single agent** — one LLM call does everything
2. **3-stage pipeline** — Researcher → Fact-Checker → Summarizer
3. **5-stage pipeline** — adds Organizer and Editor before Summarizer

We'll measure quality (via LLM judge), token cost, and execution time.

In [ ]:
# Approach 1: Single agent doing everything
single_agent = AgentNode(
    role="single_agent",
    system_prompt="""You are a comprehensive research assistant. For the given topic:
1. Research the topic thoroughly
2. Verify your facts are accurate
3. Organize the information clearly
4. Write a polished, well-structured summary
5. Keep it concise but complete (200-300 words)

Produce a final, publication-ready summary.""",
    max_tokens=600,
    temperature=0.6,
)

# Approach 2: 3-stage pipeline (already built — research_pipeline)

# Approach 3: 5-stage pipeline
organizer = AgentNode(
    role="organizer",
    system_prompt="""You are an information organizer. Take the research findings and:
1. Group related facts together
2. Arrange in logical order (most important first)
3. Remove redundancies
4. Add clear section headers
Output the organized version of the research.""",
    max_tokens=600,
    temperature=0.4,
)

pipeline_editor = AgentNode(
    role="pipeline_editor",
    system_prompt="""You are a writing editor. Polish the organized research:
1. Improve sentence flow and readability
2. Ensure consistent tone throughout
3. Fix any grammatical or stylistic issues
4. Make transitions between sections smooth
Output the edited version.""",
    max_tokens=600,
    temperature=0.5,
)

pipeline_5_stage = Pipeline(
    name="5-Stage Research Pipeline",
    stages=[researcher, fact_checker, organizer, pipeline_editor, summarizer],
)

# Run all three on the same question
test_question = Message(
    content="What are the main arguments for and against nuclear energy as a solution "
            "to climate change? Include safety, cost, and environmental considerations.",
    source_agent="user",
)

print("━" * 60)
print("APPROACH 1: Single Agent")
print("━" * 60)
single_start = time.time()
single_result = single_agent.process(test_question)
single_time = time.time() - single_start
print(f"Time: {single_time:.2f}s | Words: {len(single_result.content.split())}")
print(single_result.content[:400])

print(f"\n{'━' * 60}")
print("APPROACH 2: 3-Stage Pipeline")
print("━" * 60)
pipeline_3_result = research_pipeline.run(test_question)
print(f"Words: {len(pipeline_3_result.final_output.content.split())}")
print(pipeline_3_result.final_output.content[:400])

print(f"\n{'━' * 60}")
print("APPROACH 3: 5-Stage Pipeline")
print("━" * 60)
pipeline_5_result = pipeline_5_stage.run(test_question)
print(f"Words: {len(pipeline_5_result.final_output.content.split())}")
print(pipeline_5_result.final_output.content[:400])

### 9.1 — LLM-as-Judge Quality Comparison

We use the same LLM (with a judge prompt) to evaluate all three outputs on the same rubric.

In [ ]:
def judge_output(text: str, topic: str) -> Dict[str, Any]:
    """Use LLM as a judge to score an output."""
    judge_prompt = f"""You are an impartial quality judge. Rate the following text about "{topic}".

Score each dimension from 1 (poor) to 5 (excellent):
1. ACCURACY — Are the facts correct and well-supported?
2. COMPLETENESS — Are all important aspects covered?
3. CLARITY — Is the text clear and well-written?
4. BALANCE — Are multiple perspectives presented fairly?
5. CONCISENESS — Is the text appropriately concise without losing key info?

Respond in this exact JSON format:
{{"accuracy": N, "completeness": N, "clarity": N, "balance": N, "conciseness": N, "total": N, "brief_comment": "..."}}

Text to evaluate:
{text}"""

    messages = [
        {"role": "system", "content": "You are a fair and consistent text quality judge. Always respond in valid JSON."},
        {"role": "user", "content": judge_prompt},
    ]
    raw = generate(messages, max_new_tokens=200, temperature=0.2, do_sample=False)

    try:
        json_match = re.search(r'\{[\s\S]*\}', raw)
        if json_match:
            return json.loads(json_match.group())
    except (json.JSONDecodeError, AttributeError):
        pass
    return {"accuracy": 3, "completeness": 3, "clarity": 3, "balance": 3, "conciseness": 3, "total": 15, "brief_comment": "Parse error — default scores"}

topic = "nuclear energy and climate change"
print("Judging outputs... (this takes a minute)\n")

score_single = judge_output(single_result.content, topic)
score_3stage = judge_output(pipeline_3_result.final_output.content, topic)
score_5stage = judge_output(pipeline_5_result.final_output.content, topic)

# Display comparison table
print(f"{'Dimension':<16} {'Single':>8} {'3-Stage':>8} {'5-Stage':>8}")
print("─" * 44)
for dim in ["accuracy", "completeness", "clarity", "balance", "conciseness"]:
    s1 = score_single.get(dim, "?")
    s3 = score_3stage.get(dim, "?")
    s5 = score_5stage.get(dim, "?")
    print(f"{dim:<16} {s1:>8} {s3:>8} {s5:>8}")
print("─" * 44)
print(f"{'TOTAL':<16} {score_single.get('total', '?'):>8} {score_3stage.get('total', '?'):>8} {score_5stage.get('total', '?'):>8}")
print(f"\nTime:            {single_time:>7.1f}s {pipeline_3_result.total_time:>7.1f}s {pipeline_5_result.total_time:>7.1f}s")
print(f"\nComments:")
print(f"  Single:  {score_single.get('brief_comment', 'N/A')}")
print(f"  3-Stage: {score_3stage.get('brief_comment', 'N/A')}")
print(f"  5-Stage: {score_5stage.get('brief_comment', 'N/A')}")

## 10. When Pipelines Hurt — Anti-Patterns and Pitfalls

Pipelines aren't always better. Here are the key failure modes:

### 10.1 — Too Many Stages (Over-Engineering)

More stages = more latency, more tokens, more chances for information loss. A 10-stage pipeline for a simple question is like building a factory to make one sandwich.

**Rule of thumb**: If a single agent can do the job in one shot with acceptable quality, don't pipeline it.

### 10.2 — Information Loss Between Stages

Each stage sees only the previous stage's output, not the original input. Information gets **filtered, compressed, and reinterpreted** at each handoff:

```
Original question: "What's the GDP of France and Germany, and how do they compare?"

Stage 1 output: "France: $2.78T, Germany: $4.07T. Germany's economy is larger..."
Stage 2 output: "The European economic comparison shows significant differences..."
Stage 3 output: "Europe has diverse economic landscapes..."  ← Original question is LOST
```

### 10.3 — Overhead for Simple Tasks

| Task Complexity | Recommended Approach |
|---|---|
| Simple Q&A | Single agent |
| Moderate analysis | 2-3 stage pipeline |
| Complex research | 3-5 stage pipeline |
| Multi-format output | Conditional pipeline |

### 10.4 — Error Amplification

If Stage 1 produces a subtle error, Stage 2 may not catch it but instead build on it. Each stage **amplifies** the error rather than correcting it. This is why a fact-checking stage is valuable — it explicitly looks for errors rather than building on assumptions.

In [ ]:
# Demonstrate information loss through too many stages
def make_paraphrase_agent(stage_num: int) -> AgentNode:
    return AgentNode(
        role=f"paraphraser_{stage_num}",
        system_prompt=f"""You are paraphraser #{stage_num}. Rewrite the given text in your own words.
Maintain the same meaning but use completely different phrasing.
Do not add or remove information.""",
        max_tokens=400,
        temperature=0.7,
    )

# Build a 5-stage paraphrase pipeline (intentionally wasteful)
telephone_pipeline = Pipeline(
    name="Telephone Game (Information Loss Demo)",
    stages=[make_paraphrase_agent(i) for i in range(1, 6)],
)

original = Message(
    content="The James Webb Space Telescope, launched on December 25, 2021, cost $10 billion "
            "and orbits the Sun at the L2 Lagrange point, 1.5 million km from Earth. Its "
            "6.5-meter primary mirror collects infrared light from the universe's first galaxies, "
            "formed approximately 13.5 billion years ago.",
    source_agent="user",
)

telephone_result = telephone_pipeline.run(original)

print("\n📉 INFORMATION LOSS ANALYSIS")
print("=" * 60)
print(f"\nOriginal ({len(original.content.split())} words):")
print(original.content)
print(f"\nAfter 5 paraphrases ({len(telephone_result.final_output.content.split())} words):")
print(telephone_result.final_output.content)
print("\n⚠ Notice how specific facts (dates, costs, distances) may drift or be lost!")

## 11. Pipeline Design Guidelines — Summary

### When to Use Pipelines

✅ **Use pipelines when:**
- The task has distinct phases (research → verify → summarize)
- You need quality checks between stages
- Debugging and transparency matter
- Different stages need different LLM parameters (temperature, max tokens)
- You want to reuse stages across different workflows

❌ **Avoid pipelines when:**
- The task is simple enough for a single prompt
- Latency is critical (each stage adds a full LLM call)
- Information loss between stages would degrade quality
- The stages are so tightly coupled they can't be separated

### Design Principles

1. **Each stage should have a clear, focused responsibility** — if you can't describe a stage's job in one sentence, it's too broad
2. **Outputs should be self-contained** — each stage's output should make sense without seeing the original input
3. **Include validation stages** — fact-checking, quality scoring, format verification
4. **Log everything** — intermediate results are pipelines' biggest advantage over monolithic agents
5. **Start simple** — begin with 2-3 stages, add more only if quality measurably improves

## 12. Reusable Pipeline Patterns

A factory function makes it easy to create pre-configured pipelines for common tasks.

In [ ]:
def create_research_pipeline(
    topic_hint: str = "",
    include_fact_check: bool = True,
    summary_max_words: int = 200,
) -> Pipeline:
    """Factory for creating customized research pipelines."""
    stages = []

    context = f" Focus area: {topic_hint}." if topic_hint else ""
    stages.append(AgentNode(
        role="researcher",
        system_prompt=f"You are a thorough researcher.{context} "
                      "Find comprehensive information on the given topic. "
                      "Include facts, figures, and multiple perspectives. "
                      "Organize with clear headers and bullet points.",
        max_tokens=700,
        temperature=0.7,
    ))

    if include_fact_check:
        stages.append(AgentNode(
            role="fact_checker",
            system_prompt="You are a fact-checker. Verify claims in the text. "
                          "Mark each as VERIFIED, PLAUSIBLE, or QUESTIONABLE. "
                          "Output a corrected version with issues flagged.",
            max_tokens=700,
            temperature=0.3,
        ))

    stages.append(AgentNode(
        role="summarizer",
        system_prompt=f"Summarize the research into key points. "
                      f"Keep under {summary_max_words} words. "
                      "Prioritize accuracy and clarity. End with a one-line takeaway.",
        max_tokens=400,
        temperature=0.5,
    ))

    return Pipeline(name=f"Research: {topic_hint or 'General'}", stages=stages)

# Create specialized pipelines
science_pipeline = create_research_pipeline("science and technology")
history_pipeline = create_research_pipeline("history", include_fact_check=True)
quick_pipeline = create_research_pipeline(include_fact_check=False, summary_max_words=100)

print(f"Science: {science_pipeline}")
print(f"History: {history_pipeline}")
print(f"Quick:   {quick_pipeline}")

## 13. Exercise: Build Your Own Pipeline

Design a pipeline for one of these tasks:

1. **Code Review Pipeline**: Code Analyzer → Bug Finder → Improvement Suggester → Report Writer
2. **Debate Pipeline**: Pro-Argument Agent → Counter-Argument Agent → Synthesis Agent
3. **Translation Pipeline**: Translator → Cultural Adapter → Quality Checker

Think about:
- What system prompt does each stage need?
- What temperature is appropriate for each stage?
- Where should validation happen?
- What information might be lost between stages?

In [ ]:
# Exercise: Build a debate pipeline
# This creates a balanced analysis by forcing separate pro and con stages

pro_agent = AgentNode(
    role="pro_advocate",
    system_prompt="""You argue IN FAVOR of the proposition given.
Present the 3 strongest arguments supporting this position.
Use evidence and logical reasoning. Be persuasive but honest.
Do NOT mention counterarguments — that's another agent's job.""",
    max_tokens=500,
    temperature=0.7,
)

con_agent = AgentNode(
    role="con_advocate",
    system_prompt="""You received arguments IN FAVOR of a proposition.
Now argue AGAINST the proposition. For each pro argument presented,
provide a specific counterargument. Also add 2 new arguments against.
Be rigorous and evidence-based.""",
    max_tokens=500,
    temperature=0.7,
)

synthesis_agent = AgentNode(
    role="synthesizer",
    system_prompt="""You received pro and con arguments on a topic.
Create a balanced synthesis:
1. Summarize the strongest points from each side
2. Identify where both sides agree
3. Note which arguments are strongest based on evidence
4. Provide a nuanced conclusion that acknowledges complexity
Be fair and balanced. Do not take sides.""",
    max_tokens=500,
    temperature=0.5,
)

debate_pipeline = Pipeline("Debate Pipeline", [pro_agent, con_agent, synthesis_agent])

debate_input = Message(
    content="Should universities require all students to take artificial intelligence courses, "
            "regardless of their major?",
    source_agent="user",
)

debate_result = debate_pipeline.run(debate_input)
debate_result.show_flow()

## 14. Summary — Key Takeaways

### What We Built
1. **Message** — standardized inter-agent communication with metadata and provenance
2. **AgentNode** — specialized agent wrapper with role-specific prompts and parameters
3. **Pipeline** — sequential chain with logging, retry, and partial execution
4. **ConditionalPipeline** — dynamic routing based on content classification
5. **ValidatedAgentNode** — output validation to catch quality failures

### Core Principles

| Principle | Why It Matters |
|---|---|
| **Specialization** | Focused prompts produce better results than "do everything" prompts |
| **Interface contracts** | Standard Message format enables mix-and-match stage composition |
| **Observability** | Intermediate results make debugging possible |
| **Graceful degradation** | Partial results, retries, and error handling keep pipelines robust |

### When to Use What

- **Simple task** → Single agent call
- **Quality-sensitive task** → 2-3 stage pipeline with fact-checking
- **Complex multi-format task** → Conditional pipeline with routing
- **Research/analysis** → Research → Verify → Summarize pipeline

### Connection to Next Notebook
In **Notebook 20 (Hierarchical Agent Delegation)**, we move from **linear chains** to **tree structures** — a manager agent that decomposes tasks and delegates to specialized workers. Pipelines are one-dimensional; hierarchies handle multi-dimensional tasks.

---
*Notebook 19 of the Castalia Agents course. Next: Hierarchical Agent Delegation.*